<a href="https://colab.research.google.com/github/rayhan-khan-dev/AuraGuard-Smart-Monitor/blob/main/multi_tool_ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install pandas langchain langchain-community langchain-openai tape-agents datasets

ERROR: Could not find a version that satisfies the requirement tape-agents (from versions: none)
ERROR: No matching distribution found for tape-agents


In [17]:
import sqlite3
import pandas as pd
from datasets import load_dataset

dataset_paths = {
    "institutions": "Mahadih534/Institutional-Information-of-Bangladesh",
    "hospitals": "Mahadih534/all-bangladeshi-hospitals",
    "restaurants": "Mahadih534/Bangladeshi-Restaurant-Data"
}

def load_hf_to_sqlite(repo_id, db_name, table_name):
    try:
        print(f"Loading {repo_id} from HuggingFace")

        dataset = load_dataset(repo_id, split="train")


        df = pd.DataFrame(dataset)

        df.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]


        conn = sqlite3.connect(f"{db_name}.db")
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        conn.close()

        print(f"Success: Created '{db_name}.db' with table '{table_name}'.")
        print(f"Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Error processing {db_name}: {e}\n")

load_hf_to_sqlite(dataset_paths["institutions"], "institutions", "institutions")
load_hf_to_sqlite(dataset_paths["hospitals"], "hospitals", "hospitals")
load_hf_to_sqlite(dataset_paths["restaurants"], "restaurants", "restaurants")

Loading Mahadih534/Institutional-Information-of-Bangladesh from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Success: Created 'institutions.db' with table 'institutions'.
Columns: ['institute_name', 'eiin', 'institute_type', 'division_id', 'division', 'district_id', 'district', 'thana_id', 'thana', 'union_id', 'union_name', 'mauza_id', 'mauza_name', 'area_status', 'geogrpycal_status', 'address', 'post', 'management_type', 'mobile', 'student_type', 'education_level', 'affiliation', 'mpo_status']

Loading Mahadih534/all-bangladeshi-hospitals from HuggingFace...
Success: Created 'hospitals.db' with table 'hospitals'.
Columns: ['id', 'name', 'name_(bangla)', 'code', 'agency', 'type', 'division', 'district', 'city_corporation', 'upazila', 'paurasava', 'union', 'private']

Loading Mahadih534/Bangladeshi-Restaurant-Data from HuggingFace...
Success: Created 'restaurants.db' with table 'restaurants'.
Columns: ['place_id', 'name', 'latitude', 'longitude', 'rating', 'number_of_reviews', 'affluence', 'address']



In [18]:
import sqlite3
from langchain_core.tools import tool

@tool
def InstitutionsDBTool(sql_query: str) -> str:
    """
    Queries the institutions database to answer questions about universities,
    colleges, schools, government institutions, and board info in Bangladesh.
    Input should be a valid SQLite query for the 'institutions' table.
    Available columns: institute_name, institute_type, division, district, thana, management_type, education_level, mpo_status.
    """
    try:
        conn = sqlite3.connect("institutions.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"


@tool
def HospitalsDBTool(sql_query: str) -> str:
    """
    Queries the hospitals database to answer questions about hospitals,
    clinics, healthcare facilities, locations, and types (private/public) in Bangladesh.
    Input should be a valid SQLite query for the 'hospitals' table.
    Available columns: id, name, type, division, district, upazila, union, private.
    """
    try:
        conn = sqlite3.connect("hospitals.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"

@tool
def RestaurantsDBTool(sql_query: str) -> str:
    """
    Queries the restaurants database to answer questions about restaurants,
    food places, locations, ratings, and reviews in Bangladesh.
    Input should be a valid SQLite query for the 'restaurants' table.
    Available columns: place_id, name, latitude, longitude, rating, number_of_reviews, affluence, address.
    """
    try:
        conn = sqlite3.connect("restaurants.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"


db_tools = [InstitutionsDBTool, HospitalsDBTool, RestaurantsDBTool]
print("Custom DB Tools successfully created!")

Custom DB Tools successfully created!


In [19]:
pip install -U duckduckgo-search

In [3]:
pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.7 MB/s eta 0:00:00


In [17]:
import os
import getpass
import sqlite3
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun


if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your API key: ")


@tool
def InstitutionsDBTool(sql_query: str) -> str:
    """
    Queries the institutions database to answer questions about universities,
    colleges, schools, government institutions, and board info in Bangladesh.
    Input MUST be a valid SQLite query for the 'institutions' table.
    Available columns: institute_name, institute_type, division, district, thana, management_type, education_level, mpo_status.
    """
    try:
        conn = sqlite3.connect("institutions.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"

@tool
def HospitalsDBTool(sql_query: str) -> str:
    """
    Queries the hospitals database to answer questions about hospitals,
    clinics, healthcare facilities, locations, and types (private/public) in Bangladesh.
    Input MUST be a valid SQLite query for the 'hospitals' table.
    Available columns: id, name, type, division, district, upazila, union, private.
    """
    try:
        conn = sqlite3.connect("hospitals.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"

@tool
def RestaurantsDBTool(sql_query: str) -> str:
    """
    Queries the restaurants database to answer questions about restaurants,
    food places, locations, ratings, and reviews in Bangladesh.
    Input MUST be a valid SQLite query for the 'restaurants' table.
    Available columns: place_id, name, latitude, longitude, rating, number_of_reviews, affluence, address.
    """
    try:
        conn = sqlite3.connect("restaurants.db")
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        conn.close()
        return str(rows) if rows else "No records found."
    except Exception as e:
        return f"Error executing query: {e}"


try:
    search = DuckDuckGoSearchRun()
except Exception:
    search = None

@tool
def WebSearchTool(query: str) -> str:
    """
    Useful for answering general knowledge questions, healthcare policies,
    government regulations, definitions, or anything that requires real-time internet search
    and is not present in the local databases.
    """
    if not search:
        return "DuckDuckGo search package is missing or failed to initialize."
    try:
        return search.run(query)
    except Exception as e:
        return f"Error during web search: {e}"

all_tools = [InstitutionsDBTool, HospitalsDBTool, RestaurantsDBTool, WebSearchTool]


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

def agent_executor(user_input: str):
    print(f"\n[User Query]: {user_input}")

    response = llm_with_tools.invoke(user_input)

    if response.tool_calls:
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            print(f"🎯 [Agent Thought]: Routing to tool '{tool_name}' with arguments: {tool_args}")

            selected_tool = next((t for t in all_tools if t.name == tool_name), None)
            if selected_tool:

                if isinstance(tool_args, dict) and "sql_query" in tool_args:
                    actual_arg = tool_args["sql_query"]
                elif isinstance(tool_args, dict) and "query" in tool_args:
                    actual_arg = tool_args["query"]
                else:
                    actual_arg = str(tool_args)

                tool_output = selected_tool.invoke(actual_arg)
                print(f"📊 [Tool Output/Observation]: {tool_output}")


                final_prompt = f"""User Query: {user_input}
                Tool Used: {tool_name}
                Tool Result: {tool_output}

                Based on the tool result, provide a clear, natural language final answer to the user in English.
                """
                final_response = llm.invoke(final_prompt)
                return final_response.content
    else:
        return response.content

print("🚀 Gemini AI Agent System is successfully loaded!")

🚀 Gemini AI Agent System is successfully loaded!


In [11]:
output = agent_executor("How many hospitals are in Dhaka?")
print("\n[Final Answer]:", output)


[User Query]: How many hospitals are in Dhaka?
🎯 [Agent Thought]: Routing to tool 'HospitalsDBTool' with arguments: {'sql_query': 'SELECT COUNT(*) FROM hospitals WHERE district = "Dhaka"'}
📊 [Tool Output/Observation]: [(2919,)]

[Final Answer]: There are 2919 hospitals in Dhaka.


In [12]:
output_web = agent_executor("What is the role of DGHS in Bangladesh?")
print("\n[Final Answer]:", output_web)


[User Query]: What is the role of DGHS in Bangladesh?
🎯 [Agent Thought]: Routing to tool 'WebSearchTool' with arguments: {'query': 'role of DGHS in Bangladesh'}
📊 [Tool Output/Observation]: DuckDuckGo search package is missing or failed to initialize.

[Final Answer]: I apologize, but the WebSearchTool encountered an error ("DuckDuckGo search package is missing or failed to initialize") and was unable to retrieve information about the role of DGHS in Bangladesh. Therefore, I cannot provide an answer to your question at this time.


In [15]:
@tool
def WebSearchTool(query: str) -> str:
    """
    Useful for answering general knowledge questions, healthcare policies,
    government regulations, definitions, or anything that requires real-time internet search
    and is not present in the local databases.
    """

    try:
        from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
        search_wrapper = DuckDuckGoSearchAPIWrapper()
        return search_wrapper.run(query)
    except Exception:

        print("⚠️ [WebSearchTool Notice]: Live web package failed, using fallback LLM knowledge base.")
        fallback_prompt = f"Provide a brief, accurate summary for this general knowledge query: {query}"
        fallback_res = llm.invoke(fallback_prompt)
        return fallback_res.content

all_tools = [InstitutionsDBTool, HospitalsDBTool, RestaurantsDBTool, WebSearchTool]
llm_with_tools = llm.bind_tools(all_tools)
print("✅ Fully robust WebSearchTool is ready!")

✅ Fully robust WebSearchTool is ready!


In [16]:
output_web = agent_executor("What is the role of DGHS in Bangladesh?")
print("\n[Final Answer]:", output_web)


[User Query]: What is the role of DGHS in Bangladesh?
🎯 [Agent Thought]: Routing to tool 'WebSearchTool' with arguments: {'query': 'role of DGHS in Bangladesh'}
⚠️ [WebSearchTool Notice]: Live web package failed, using fallback LLM knowledge base...
📊 [Tool Output/Observation]: The Directorate General of Health Services (DGHS) is the principal government organization responsible for managing and delivering public health services in Bangladesh.

Operating under the Ministry of Health and Family Welfare, its primary role encompasses the planning, implementation, monitoring, and evaluation of all health programs and policies across the country. This includes managing public hospitals and clinics, disease surveillance and control, maternal and child health, primary healthcare, and health promotion initiatives. Essentially, DGHS is central to ensuring the accessibility, quality, and equity of healthcare services for the entire population of Bangladesh.

[Final Answer]: The Directorate Gene